# 5.6. GPU

本节介绍如何把计算分配到 GPU，实现加速。请自己完成每个代码单元。

查看显卡信息（可在 CPU 环境运行验证，不强制）：
`!nvidia-smi`

导入 torch、torch.nn。

In [7]:
import torch
from torch import nn

### 5.6.1. 计算设备

在 PyTorch 中，CPU 和 GPU 用 torch.device 表示：
- torch.device('cpu'), torch.device('cuda'), torch.device('cuda:1')

查询可用 GPU 数量：
- torch.cuda.device_count()

In [9]:
torch.cuda.device_count()

1

定义两个便捷函数，使代码在无 GPU 时也能运行：
- def try_gpu(i=0): return torch.device(f'cuda:{i}') if torch.cuda.device_count() >= i + 1 else torch.device('cpu')
- def try_all_gpus(): devices = [torch.device(f'cuda:{i}') for i in range(torch.cuda.device_count())]；return devices if devices else [torch.device('cpu')]

（也可使用 d2l.torch 中的 d2l.try_gpu / d2l.try_all_gpus）

In [19]:
def try_gpu(i=0):
    if torch.cuda.device_count()>=i+1:
        return torch.device(f'cuda:{i}')
    else:
        return torch.device('cpu')

### 5.6.2. 张量与 GPU

默认张量在 CPU 上，可通过 .to(dev) 或 .cuda() 移动到 GPU：
- X = torch.tensor([1, 2, 3])；X.device 查看所在设备
- X.cuda(0) 移动到 GPU（若支持）；再用 X.cpu() 移回 CPU
- 想办法打印张量当前所在设备（X.device）

In [20]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
X=torch.tensor([1,2,3])
X=X.to(device)
print(X.device)

cuda:0


### 5.6.3. 神经网络与 GPU

将网络放到 GPU 上，并检查参数所在设备：
- net = nn.Sequential(nn.Linear(256, 256))；用 try_gpu() 获取设备
- net = net.to(dev)；查看 net[0].weight.data.device
- 送入 GPU 上的数据 X = torch.rand(size=(2, 256), device=dev)；net(X)

In [25]:
net=nn.Sequential(nn.Linear(256,256))
dev=try_gpu()
net=net.to(dev)
net[0].weight.data.device
X=torch.rand(2,256)
X=X.to(dev)
net(X)

tensor([[ 2.7022e-01,  5.3022e-01, -1.5299e-01, -2.4957e-01, -1.9769e-01,
          6.9624e-02,  8.4670e-02, -6.2983e-02,  4.8022e-02, -2.1023e-01,
         -1.0865e-01, -1.3089e-01,  1.0429e-01,  3.3704e-01, -3.0121e-01,
          4.8900e-01, -3.2169e-02,  2.3896e-01,  2.5832e-01,  3.8390e-01,
         -7.6505e-02,  3.2255e-01, -1.3000e-01,  4.4770e-01, -1.7861e-01,
         -5.6381e-01, -3.1034e-01,  9.7322e-03,  5.8329e-01,  5.7903e-01,
          5.2096e-02,  4.7321e-02, -1.4680e-01, -2.3491e-01, -2.1678e-01,
          5.5321e-01, -1.0034e-01,  4.6990e-01, -2.8218e-01, -3.5755e-01,
          1.5737e-01, -2.1713e-02,  2.4560e-01,  2.3121e-01, -3.9463e-02,
         -6.3734e-01,  4.0913e-04, -1.3360e-01, -8.5042e-02,  4.1496e-01,
          3.3495e-01,  3.5505e-01,  2.4073e-01, -1.0718e-01,  4.6177e-01,
         -5.6092e-01,  2.5133e-01, -1.8222e-01, -3.0300e-01, -5.2375e-01,
         -4.2727e-01,  3.1389e-01,  2.3105e-01, -2.4637e-01, -3.8070e-01,
          6.7575e-01,  1.5084e-01,  1.

### 5.6.4. 小结

- 可通过 torch.device 指定计算设备（CPU / CUDA）
- 张量与模型可用 .to() / .cuda() 在设备间移动
- 合理分配设备可减少数据传输开销，显著加速训练

### 5.6.5. 练习

1. 尝试一个较大的张量计算，比较 CPU 与 GPU 的执行速度。
2. 若主机只有 CPU，代码是否能通过 try_gpu 正常运行？
3. 尝试读取显卡上某张量的数值（如子集、精度打印）。

In [31]:
x1=torch.rand(10005,10005)
x2=torch.rand(10005,10005)
print(x1@x2)


tensor([[2504.5579, 2493.1118, 2472.0049,  ..., 2522.6299, 2517.1262,
         2504.2686],
        [2531.8921, 2504.3477, 2503.2695,  ..., 2516.1453, 2527.2510,
         2530.4104],
        [2523.1216, 2508.7822, 2508.2112,  ..., 2511.8564, 2521.8120,
         2526.3525],
        ...,
        [2516.1189, 2531.2253, 2507.9321,  ..., 2541.9094, 2530.5654,
         2532.0000],
        [2521.3567, 2518.9800, 2499.4231,  ..., 2514.0627, 2530.5903,
         2502.8157],
        [2517.1633, 2513.9333, 2502.9883,  ..., 2530.1006, 2515.5359,
         2517.5759]])


In [32]:
x1=x1.to(dev)
x2=x2.to(dev)
print(x1@x2)

tensor([[2504.5574, 2493.1123, 2472.0061,  ..., 2522.6248, 2517.1223,
         2504.2678],
        [2531.8831, 2504.3469, 2503.2734,  ..., 2516.1487, 2527.2551,
         2530.4072],
        [2523.1182, 2508.7827, 2508.2185,  ..., 2511.8596, 2521.8057,
         2526.3462],
        ...,
        [2516.1206, 2531.2280, 2507.9285,  ..., 2541.9102, 2530.5645,
         2531.9988],
        [2521.3638, 2518.9761, 2499.4255,  ..., 2514.0623, 2530.5876,
         2502.8169],
        [2517.1675, 2513.9407, 2502.9910,  ..., 2530.0969, 2515.5386,
         2517.5713]], device='cuda:0')
